In [44]:
from sqlalchemy import create_engine
# from sqlalchemy.pool import NullPool
from dotenv import load_dotenv
import os

In [45]:


# Load environment variables from .env
load_dotenv()

# Fetch variables
USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

# Construct the SQLAlchemy connection string
DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

# Create the SQLAlchemy engine
engine = create_engine(DATABASE_URL)
# If using Transaction Pooler or Session Pooler, we want to ensure we disable SQLAlchemy client side pooling -
# https://docs.sqlalchemy.org/en/20/core/pooling.html#switching-pool-implementations
# engine = create_engine(DATABASE_URL, poolclass=NullPool)

# Test the connection
try:
    with engine.connect() as connection:
        print("Connection successful!")
except Exception as e:
    print(f"Failed to connect: {e}")

Failed to connect: (psycopg2.OperationalError) could not translate host name "None" to address: nodename nor servname provided, or not known

(Background on this error at: https://sqlalche.me/e/21/e3q8)


In [46]:
import pandas as pd

table_name = "proj2_sp500"
df = pd.read_sql_table(table_name, engine)

display(df.head())


OperationalError: (psycopg2.OperationalError) could not translate host name "None" to address: nodename nor servname provided, or not known

(Background on this error at: https://sqlalche.me/e/21/e3q8)

In [52]:
# Kuoshen Start
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

# 1. Force reload the updated .env file (wiping old memory cache)
load_dotenv(override=True)

# 2. Fetch your BigQuery project credentials
BILLING_PROJECT = os.getenv("BILLING_PROJECT_ID")
DATASET = os.getenv("DATASET_ID")

# 3. Construct the official BigQuery dialect connection string
# Format: bigquery://billing-project-id/dataset-id
DATABASE_URL = f"bigquery://{BILLING_PROJECT}/{DATASET}"

# 4. Initialize the BigQuery SQLAlchemy engine
engine = create_engine(DATABASE_URL)

# 5. Test the connection sequence
try:
    with engine.connect() as connection:
        print("🎉 Successfully connected to Google BigQuery!")
except Exception as e:
    print(f"❌ Failed to connect to BigQuery: {e}")


🎉 Successfully connected to Google BigQuery!


In [53]:
import pandas as pd

# Define your targeted dbt transformed model or fact table name
# Replace 'your_table_name' with the actual output table created by dbt
table_name = "public_proj2_sp500" 

# Construct a standard GoogleSQL query string 
query = f"SELECT * FROM `{BILLING_PROJECT}.{DATASET}.{table_name}` LIMIT 1000"

# Pull the remote data records straight into a local Pandas memory block
df = pd.read_sql(query, con=engine)

# View structural sizing and verify column fields loaded correctly
print(f"DataFrame Shape: {df.shape}")
print(df.head())


DataFrame Shape: (1000, 46)
         date ticker   company                  sector sub_industry  \
0  2021-12-09    APP  AppLovin  Communication Services  Advertising   
1  2021-12-15    APP  AppLovin  Communication Services  Advertising   
2  2021-12-20    APP  AppLovin  Communication Services  Advertising   
3  2021-12-28    APP  AppLovin  Communication Services  Advertising   
4  2021-12-29    APP  AppLovin  Communication Services  Advertising   

   date_added   open    high    low  close  ...  liabilities_ps  shares_out  \
0  2025-09-22  96.00  99.295  90.05  90.78  ...          6.7405   373641135   
1  2025-09-22  86.98  89.120  84.93  88.56  ...          6.7405   373641135   
2  2025-09-22  86.26  89.680  86.01  87.36  ...          6.7405   373641135   
3  2025-09-22  95.28  96.770  92.71  93.19  ...          6.7405   373641135   
4  2025-09-22  92.73  93.060  88.68  91.28  ...          6.7405   373641135   

        id                _sdc_extracted_at                 _sdc_recei

In [54]:
# to check for table name error 

import pandas as pd

# Query the active metadata framework map
list_tables_query = f"""
SELECT table_name, table_type 
FROM `{BILLING_PROJECT}.{DATASET}.INFORMATION_SCHEMA.TABLES`
"""

# Fetch the structural mapping directly into a DataFrame
tables_df = pd.read_sql(list_tables_query, con=engine)

print("✨ Available tables/views sitting in your data warehouse:")
print(tables_df)


✨ Available tables/views sitting in your data warehouse:
           table_name  table_type
0     fact_dailyprice  BASE TABLE
1  public_proj2_sp500  BASE TABLE
2          dim_ticker  BASE TABLE


In [62]:
import pandas as pd


# 1. Load the 600k row daily price fact table (Limiting to 100,000 for a swift initial check)
fact_query = f"SELECT * FROM `{BILLING_PROJECT}.{DATASET}.fact_dailyprice` LIMIT 100000"
df_fact = pd.read_sql(fact_query, con=engine)

# 2. Load the ticker company dimension table
dim_query = f"SELECT * FROM `{BILLING_PROJECT}.{DATASET}.dim_ticker`"
df_dim = pd.read_sql(dim_query, con=engine)

print(f"🎉 Fact Table Loaded: {df_fact.shape}")
print(f"🎉 Dimension Table Loaded: {df_dim.shape}")


🎉 Fact Table Loaded: (100000, 41)
🎉 Dimension Table Loaded: (503, 4)


In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(override=True)
BILLING_PROJECT = os.getenv("BILLING_PROJECT_ID")
DATASET = os.getenv("DATASET_ID")

# 1. Re-create the engine but explicitly DISABLE the storage client API hooks
DATABASE_URL = f"bigquery://{BILLING_PROJECT}/{DATASET}"
engine = create_engine(
    DATABASE_URL, 
    # This configuration forces the driver to run standard SQL queries 
    # instead of initiating Storage Read API sessions
    connect_args={"use_bqstorage_api": False}
)

# 2. Start with a smaller initial subset row limit to verify it works safely
fact_query = f"SELECT * FROM `{BILLING_PROJECT}.{DATASET}.fact_dailyprice` LIMIT 10000"
df_fact = pd.read_sql(fact_query, con=engine)

# 3. Load the dimension table
dim_query = f"SELECT * FROM `{BILLING_PROJECT}.{DATASET}.dim_ticker`"
df_dim = pd.read_sql(dim_query, con=engine)

print(f"🎉 Fact Table Loaded: {df_fact.shape}")
print(f"🎉 Dimension Table Loaded: {df_dim.shape}")


In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(override=True)

BILLING_PROJECT = os.getenv("BILLING_PROJECT_ID")
DATASET = os.getenv("DATASET_ID")

DATABASE_URL = f"bigquery://{BILLING_PROJECT}/{DATASET}"

engine = create_engine(DATABASE_URL)

fact_query = f"""
SELECT *
FROM `{BILLING_PROJECT}.{DATASET}.fact_dailyprice`
LIMIT 10000
"""

df_fact = pd.read_sql(fact_query, con=engine)

print(f"Fact Table Loaded: {df_fact.shape}")

In [11]:
from google.cloud import bigquery
import os

project = os.getenv("BILLING_PROJECT_ID")
dataset = os.getenv("DATASET_ID")

print("Project:", project)
print("Dataset:", dataset)

client = bigquery.Client(project=project)

query = f"""
SELECT COUNT(*) AS row_count
FROM `{project}.{dataset}.fact_dailyprice`
"""

result = client.query(query).result()

for row in result:
    print("Row count:", row.row_count)

Project: dsaimodule2-509312
Dataset: proj2sp500
Row count: 622710


In [12]:
import os

print("GOOGLE_APPLICATION_CREDENTIALS:",
      os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))

print("BILLING_PROJECT_ID:",
      os.getenv("BILLING_PROJECT_ID"))

print("DATASET_ID:",
      os.getenv("DATASET_ID"))

GOOGLE_APPLICATION_CREDENTIALS: None
BILLING_PROJECT_ID: dsaimodule2-509312
DATASET_ID: proj2sp500


In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(override=True)

BILLING_PROJECT = os.getenv("BILLING_PROJECT_ID")
DATASET = os.getenv("DATASET_ID")

DATABASE_URL = f"bigquery://{BILLING_PROJECT}/{DATASET}"

engine = create_engine(
    DATABASE_URL,
    connect_args={
        "create_bqstorage_client": False
    }
)

fact_query = f"""
SELECT *
FROM `{BILLING_PROJECT}.{DATASET}.fact_dailyprice`
LIMIT 10000
"""

df_fact = pd.read_sql(fact_query, con=engine)

print(f"Fact Table Loaded: {df_fact.shape}")